# Step 3: Cleanup — Delete All Workspaces

Run this notebook after the event to delete all team workspaces created in Step 1.

## Prerequisites

1. You must have the `teams-resolved.xlsx` file in your Lakehouse (output from Step 1)
2. You must have **Fabric workspace deletion permissions**

## Configuration

Edit the values below:

In [ ]:
# Configuration — edit these
RESULTS_FILE = "Files/teams-resolved.xlsx"  # Output from Step 1
DRY_RUN = True  # Set to False to actually delete workspaces
SKIP_CONFIRMATION = False  # Set to True to skip confirmation prompt

## Step 1: Load Workspace Data

In [ ]:
import pandas as pd

print(f"📂 Loading Lakehouse file: {RESULTS_FILE}")
df = pd.read_excel(f"/lakehouse/default/{RESULTS_FILE}")

# Validate columns
if "WorkspaceId" not in df.columns:
    raise ValueError(
        f"❌ File must have 'WorkspaceId' column\n"
        f"   Found: {df.columns.tolist()}"
    )

# Get unique workspaces
workspaces = df.drop_duplicates(subset=["WorkspaceId"])[["TeamName", "WorkspaceId"]].to_dict(orient="records")
print(f"✅ Found {len(workspaces)} workspaces to delete:")
for ws in workspaces:
    print(f"  - {ws['TeamName']}: {ws['WorkspaceId']}")

## Step 2: Get Fabric API Token

In [ ]:
import notebookutils

print("🔐 Getting Fabric API token...")
fabric_token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
fabric_headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}
print("✅ Token acquired")

## Step 3: Confirmation

In [ ]:
if not DRY_RUN and not SKIP_CONFIRMATION:
    print("⚠️  WARNING: This will DELETE all team workspaces!")
    print(f"\nWorkspaces to delete ({len(workspaces)}):")
    for ws in workspaces:
        print(f"  - {ws['TeamName']}: {ws['WorkspaceId']}")
    print("\nType 'yes' in the next cell to confirm, or just run it as-is to cancel.")
    confirmation = input("Confirm deletion (type 'yes'): ")
    if confirmation.lower() != "yes":
        print("❌ Cancelled. No workspaces deleted.")
        raise SystemExit()
else:
    print(f"{'🔍 DRY RUN MODE - No changes will be made' if DRY_RUN else '✅ Confirmation skipped (SKIP_CONFIRMATION=True)'}")

## Step 4: Delete Workspaces

In [ ]:
import requests

print(f"\n🗑️  Deleting workspaces...\n")

successful = 0
failed = 0

def delete_workspace(workspace_id: str, workspace_name: str) -> bool:
    """Delete a Fabric workspace."""
    if DRY_RUN:
        print(f"  [DRY RUN] Would delete {workspace_name}")
        return True

    response = requests.delete(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}",
        headers=fabric_headers,
    )

    if response.status_code in (200, 204):
        print(f"  ✅ Deleted {workspace_name}")
        return True
    elif response.status_code == 404:
        print(f"  ✅ {workspace_name} already deleted (not found)")
        return True
    else:
        print(f"  ❌ Failed to delete {workspace_name}: {response.status_code}")
        if response.text:
            print(f"     {response.text[:300]}")
    return False

for ws in workspaces:
    team_name = ws["TeamName"]
    workspace_id = ws["WorkspaceId"]
    if delete_workspace(workspace_id, team_name):
        successful += 1
    else:
        failed += 1

print(f"\n✨ Complete!")
print(f"  ✅ Successful: {successful}")
if failed > 0:
    print(f"  ❌ Failed: {failed}")